In [ ]:
import re
import os
import pandas as pd

def convert_vissim_rsr_to_excel(rsr_file_path, excel_file_path):
    """
    Convert .rsr file to xlsx file. This function reads a Vissim .rsr result file,
    extracts relevant data lines, and saves them into an Excel file. It also attempts to convert columns to numeric types
    to align with the pandas latest style.
    
    :param rsr_file_path: input .rsr file path
    :param excel_file_path: output .xlsx file path
    :return: True if conversion is successful, False otherwise
    """
    try:
        data_lines = []
        # Vissim result files are usually encoded in cp949.
        with open(rsr_file_path, 'r', encoding='cp949', errors='ignore') as f:
            for line in f:
                # Starting numbered lines with semicolon are data lines.
                if re.match(r'^\s*\d+\.\d+;', line.strip()):
                    data_lines.append(line.strip())

        if not data_lines:
            print(f"Not Found valid data lines in '{rsr_file_path}' file.")
            return False

        header = ['Time', 'No', 'Veh', 'VehType', 'Trav', 'Delay', 'Dist']

        parsed_data = []
        for line in data_lines:
            values = [v.strip() for v in line.split(';')]
            if values and values[-1] == '':
                values.pop()

            try:
                row_dict = {header[i]: values[i] for i in range(len(values))}
                parsed_data.append(row_dict)
            except IndexError:
                print(f"IndexError: Incorrect number of values in line: {line}")

        df = pd.DataFrame(parsed_data)

        # Attempt to convert columns to numeric types where possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except ValueError:
                # If conversion fails, keep the column as is (likely string)
                pass

        df.to_excel(excel_file_path, index=False)
        print(f"✅ Successfully converted '{rsr_file_path}' to Excel format.")
        return True

    except FileNotFoundError:
        print(f"❌ Error: '{rsr_file_path}' file not found.")
        return False
    except Exception as e:
        print(f"❌ Error: An error occurred during file conversion: {e}")
        return False

def analyze_vissim_output(excel_file_path, log_file_path='log.txt'):
    """
    Analyze the converted Excel file to compute average speeds and traffic volumes,
    and save the results to a log.txt file.

    :param excel_file_path: Path to the Excel file to be analyzed
    :param log_file_path: Path to the log file where results will be saved
    """
    try:
        df = pd.read_excel(excel_file_path)
        print(f"✅ Loaded Excel file: '{excel_file_path}'")

        # Initialize results dictionary
        results = {}

        # 1. Average Speed Calculation (kph)
        # Speed = (Distance(m) / Travel Time(s)) * 3.6
        speed_conditions = {
            'a0_speed': [1, 2, 3],
            'a1_speed': [4, 5, 6, 7, 8],
            'a2_speed': [9, 10, 11, 12, 13],
            'a3_speed': [14, 15, 16, 17, 18],
            'a4_speed': [19, 20, 21],
            'a5_speed': [22, 23, 24, 25, 26]
        }

        for name, no_list in speed_conditions.items():
            # Filter rows corresponding to the specified 'No' values
            subset = df[df['No'].isin(no_list)].copy()
            # Calculate speed only for rows where travel time (Trav) is greater than 0
            subset = subset[subset['Trav'] > 0]

            if not subset.empty:
                subset['Speed_kmh'] = (subset['Dist'] / subset['Trav']) * 3.6
                avg_speed = subset['Speed_kmh'].mean()
                results[name] = avg_speed
            else:
                results[name] = 0.0 # If no valid data, set average speed to 0

        # 2. Traffic Volume Calculation (vph)
        # Count the number of vehicles for each 'No' condition and convert to vehicles per hour (vph)
        volume_conditions = {
            'Q0_vph': [1],
            'Q1_vph': [2, 3],
            'Q2_vph': [7],
            'Q3_vph': [5],
            'Q4_vph': [4, 6, 8],
            'Q5_vph': [12, 13],
            'Q6_vph': [9, 11],
            'Q7_vph': [10],
            'Q8_vph': [17],
            'Q9_vph': [14, 16],
            'Q10_vph': [15, 18],
            'Q11_vph': [19],
            'Q12_vph': [20, 21],
            'Q13_vph': [24, 26],
            'Q14_vph': [22, 25],
            'Q15_vph': [23]
        }

        for name, no_list in volume_conditions.items():
            count = len(df[df['No'].isin(no_list)])
            results[name] = count

        # 3. Sum of Input Traffic Volumes
        results['input0'] = results.get('Q0_vph', 0) + results.get('Q1_vph', 0)
        results['input1'] = results.get('Q2_vph', 0) + results.get('Q3_vph', 0) + results.get('Q4_vph', 0)
        results['input2'] = results.get('Q5_vph', 0) + results.get('Q6_vph', 0) + results.get('Q7_vph', 0)
        results['input3'] = results.get('Q8_vph', 0) + results.get('Q9_vph', 0) + results.get('Q10_vph', 0)
        results['input4'] = results.get('Q11_vph', 0) + results.get('Q12_vph', 0)
        results['input5'] = results.get('Q13_vph', 0) + results.get('Q14_vph', 0) + results.get('Q15_vph', 0)

        # Ensure the log file directory exists
        with open(log_file_path, 'w', encoding='utf-8') as f:
            f.write(f"filename:{log_file_path} ===\n")
            f.write("--- vissim_1_no Average Speed Analysis (km/h) ---\n")
            for name, value in speed_conditions.items():
                f.write(f"average {name}: {results.get(name, 0):.2f}\n")

            f.write("\n--- Traffic Volume Analysis (vph) ---\n")
            # Print traffic volumes Q0 to Q15 in order
            for i in range(16):
                name = f"Q{i}_vph"
                log_name = name
                f.write(f"{log_name}: {results.get(name, 0)}\n")

            f.write("\n--- Output Volume Summary (vph) ---\n")
            for i in range(6):
                name = f"input{i}"
                f.write(f"Output{i}: {results.get(name, 0)}\n")

        print(f"✅ Analysis complete! Results saved to '{log_file_path}'.")

    except FileNotFoundError:
        print(f"❌ Error: The Excel file '{excel_file_path}' to be analyzed was not found.")
    except KeyError as e:
        print(f"❌ Error: The Excel file is missing required columns ({e}). Please check the file format.")
    except Exception as e:
        print(f"❌ Error: An error occurred during analysis: {e}")


## Set the .rsr input folder path

In [ ]:
test_dir = 'YOUR_RSR_FILE_DIRECTORY'

print(f"Count rsr files : {len(os.listdir(test_dir))}")

In [ ]:
if __name__ == "__main__":
  # 0. Set the .rsr input folder path
  # example: test_dir = 'YOUR_RSR_FILE_DIRECTORY'

  test_dir = test_dir
  print(f"rsr folder : {test_dir}")
  print(f"Count rsr files: {len(os.listdir(test_dir))}")

  num = 0
  for file in os.listdir(test_dir):
    input_rsr_file = os.path.join(test_dir, file)
    print(f"{num} \n✅ Input rsr file! {input_rsr_file}")

    # 1. Generate the output Excel file path automatically (.rsr -> .xlsx)
    output_excel_file = input_rsr_file.replace('.rsr', '.xlsx')

    # 2. Convert the RSR file to an Excel file.
    conversion_successful = convert_vissim_rsr_to_excel(input_rsr_file, output_excel_file)

    # 3. Run the analysis only if the conversion was successful.
    if conversion_successful:
      # 4. Save the results to log.txt from the converted Excel file.
      analyze_vissim_output(output_excel_file, log_file_path=output_excel_file.replace('.xlsx', '.txt'))
    else:
      print(f"🔥 Analysis failed! {output_excel_file}")
    num+=1